# 🧬 Hybrid Production Notebook: YOLOv11 + SAM 2 + Classification + Pothole Analysis
## Detect (YOLO) → Segment (SAM 2) → Classify (Pavement) → Measure (Real IPM)

---

**Workflow:**
1.  **Calibration**: Interactie tool to click 4 A4 corners (or Auto-detect).
2.  **Detect (YOLOv11)**: Fast bounding box localization & Tracking (ByteTrack).
3.  **Segment (SAM 2)**: Zero-shot segmentation using detected boxes as prompts.
4.  **Classify (YOLOv8-Cls)**: Identify Pavement Type (Asphalt/Concrete) from crop.
5.  **Measure (IPM)**: Warp masks to Bird's Eye View for **real-world area accuracy**.

**Output:** Web-app ready `detections.csv` (with repair cost data) and annotated images.

## 1️⃣ Setup & Installation

In [ ]:
# @title 1️⃣ Install Dependencies
# @markdown Installs YOLOv11 and SAM 2. **Restart Runtime** if prompted!
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Install YOLO
!pip install -q -U ultralytics

# Install SAM 2 (Official Facebook Repo)
!pip install -q git+https://github.com/facebookresearch/segment-anything-2.git

# Install Utils
!pip install -q opencv-python pandas numpy tqdm

# Download SAM 2 Checkpoint (Small model for speed)
!wget -q https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_small.pt

print("✅ Setup Complete.")

In [ ]:
# @title 2️⃣ Configuration & File Paths
import os

# @markdown ### 📂 File Paths
VIDEO_PATH = "/content/video.mp4" # @param {type:"string"}
GPS_LOG_PATH = "/content/gps_log.csv" # @param {type:"string"}
DET_MODEL_PATH = "/content/best_det.pt" # @param {type:"string"}
CLS_MODEL_PATH = "/content/best_cls.pt" # @param {type:"string"}
CALIB_IMAGE_PATH = "/content/calibration.jpg" # @param {type:"string"}

# @markdown ### ⚙️ Inference Settings
FRAME_SKIP = 5 # @param {type:"integer"}
CONFIDENCE_THRESHOLD = 0.25 # @param {type:"number"}
IMG_SIZE = 640 # @param {type:"integer"}

# @markdown ### 🎛️ Features
USE_TRACKING = True # @param {type:"boolean"}
GENERATE_VIDEO = False # @param {type:"boolean"}

# @markdown ### 📏 Calibration & Manual Points
# @markdown **Leave empty `[]` to use Auto-Detection**. 
# @markdown Or paste points from Tool below: `[[x1,y1], [x2,y2], [x3,y3], [x4,y4]]`
MANUAL_CALIB_POINTS = [] # @param {type:"raw"}
ROAD_WIDTH_M = 3.5 # @param {type:"number"}
CAMERA_HEIGHT_M = 1.5 # @param {type:"number"}

print(f"✅ Config Loaded. Tracking: {USE_TRACKING}, Video: {GENERATE_VIDEO}")

In [ ]:
# @title 2.5️⃣ (Tool) Interactive Calibration Helper 🖱️
# @markdown **Run this cell ONLY if you need to click points manually.**
# @markdown 1. Ensure `calibration.jpg` is uploaded.
# @markdown 2. Click 4 corners: **Top-Left, Top-Right, Bottom-Right, Bottom-Left**.
# @markdown 3. Copy the output list into `MANUAL_CALIB_POINTS` above.

import cv2
from google.colab.output import eval_js
from base64 import b64encode
import numpy as np
from IPython.display import display, Javascript, Image
from google.colab import output

def show_interactive_calib(image_path):
    if not os.path.exists(image_path):
        print("⚠️ Calibration image not found. Upload 'calibration.jpg' first.")
        return

    with open(image_path, "rb") as f:
        img_bytes = f.read()
    img_b64 = b64encode(img_bytes).decode('utf-8')
    
    js_code = f"""
    <div id="calib_div">
      <h3>Click 4 Corners (TL, TR, BR, BL)</h3>
      <canvas id="calib_canvas" style="border:1px solid red; cursor:crosshair;"></canvas>
      <p id="points_log">Points: []</p>
      <button onclick="finishCalib()">Done</button>
    </div>
    <script>
      var canvas = document.getElementById('calib_canvas');
      var ctx = canvas.getContext('2d');
      var img = new Image();
      img.src = "data:image/jpeg;base64,{img_b64}";
      var points = [];
      img.onload = function() {{
        var scale = 1.0;
        if (img.width > 800) {{ scale = 800 / img.width; }}
        canvas.width = img.width * scale;
        canvas.height = img.height * scale;
        ctx.drawImage(img, 0, 0, canvas.width, canvas.height);
        canvas.onclick = function(e) {{
          if (points.length >= 4) return;
          var rect = canvas.getBoundingClientRect();
          var x = (e.clientX - rect.left) / scale;
          var y = (e.clientY - rect.top) / scale;
          points.push([Math.round(x), Math.round(y)]);
          ctx.fillStyle = "red"; ctx.beginPath();
          ctx.arc(x * scale, y * scale, 5, 0, 2*Math.PI); ctx.fill();
          document.getElementById('points_log').innerText = "Points: " + JSON.stringify(points);
        }};
        window.finishCalib = function() {{
           google.colab.kernel.invokeFunction('notebook.ManualCalibCallback', [points], {{}});
        }};
      }};
    </script>
    """
    display(Javascript(js_code))

def calib_callback(points):
    print("\n✅ COPY THIS BELOW:")
    print(f"MANUAL_CALIB_POINTS = {points}")

output.register_callback('notebook.ManualCalibCallback', calib_callback)

if os.path.exists(CALIB_IMAGE_PATH):
    show_interactive_calib(CALIB_IMAGE_PATH)
else:
    pass # Silent fail if not exists

## 2️⃣ Core Functions (IPM & Area)

In [ ]:
import cv2
import numpy as np

def order_points(pts):
    """Orders coordinates: top-left, top-right, bottom-right, bottom-left."""
    rect = np.zeros((4, 2), dtype="float32")
    s = pts.sum(axis=1)
    rect[0] = pts[np.argmin(s)]
    rect[2] = pts[np.argmax(s)]
    diff = np.diff(pts, axis=1)
    rect[1] = pts[np.argmin(diff)]
    rect[3] = pts[np.argmax(diff)]
    return rect

def calculate_homography(calibration_image_path, manual_points=None, road_width_m=3.5):
    """
    Calculates Homography Matrix (H).
    Priority: 1. Manual Points, 2. Auto A4 Detection, 3. None (Fallback)
    A4 Dimensions: 210mm x 297mm (0.21m x 0.297m)
    """
    if not os.path.exists(calibration_image_path):
        return None, None
        
    img = cv2.imread(calibration_image_path)
    if img is None: return None, None
    
    rect = None

    # 1. Use Manual Points if provided
    if manual_points and len(manual_points) == 4:
        pts = np.array(manual_points, dtype="float32")
        rect = order_points(pts)
        print("✅ Using Manual Calibration Points.")
    
    # 2. Auto Detection
    else:
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        blur = cv2.GaussianBlur(gray, (5, 5), 0)
        thresh = cv2.adaptiveThreshold(blur, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, 
                                       cv2.THRESH_BINARY, 11, 2)
        thresh = cv2.bitwise_not(thresh)
        cnts, _ = cv2.findContours(thresh.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cnts = sorted(cnts, key=cv2.contourArea, reverse=True)
        
        paper_cnt = None
        for c in cnts:
            peri = cv2.arcLength(c, True)
            approx = cv2.approxPolyDP(c, 0.02 * peri, True)
            if len(approx) == 4:
                paper_cnt = approx
                break
        
        if paper_cnt is not None:
            pts = paper_cnt.reshape(4, 2)
            rect = order_points(pts)
            print("✅ Auto-Detected A4 Paper.")
        else:
            print("⚠️ Could not detect A4. Provide Manual Points or use Fallback.")
            return None, None

    # Destination Points (Bird's Eye View of A4)
    # Scale: 1000 pixels per meter
    scale = 1000.0 
    # A4 is 0.210m x 0.297m -> 210px x 297px
    w_px = 0.210 * scale
    h_px = 0.297 * scale
    
    dst = np.array([
        [0, 0],
        [w_px, 0],
        [w_px, h_px],
        [0, h_px]
    ], dtype="float32")
    
    H = cv2.getPerspectiveTransform(rect, dst)
    return H, scale

def calculate_area_hybrid(mask, width, height, H=None, scale=1000.0):
    """
    Calculates area using SAM Mask.
    1. If Homography (H) exists: Warp mask -> Count Pixels -> m².
    2. Else: Use Fallback geometric PPM.
    """
    pixel_count = np.sum(mask > 0.0)
    
    if H is not None:
        mask_u8 = (mask > 0.0).astype(np.uint8) * 255
        warped_mask = cv2.warpPerspective(mask_u8, H, (width*2, height*2))
        warped_pixels = cv2.countNonZero(warped_mask)
        area_m2 = warped_pixels / (scale ** 2)
        return area_m2
    else:
        assumed_road_px = width * 0.8
        ppm = assumed_road_px / ROAD_WIDTH_M
        area_m2 = pixel_count / (ppm ** 2)
        return area_m2

## 3️⃣ Load Models (Detect + Segment + Classify)

In [ ]:
from ultralytics import YOLO
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. YOLOv11 Detection
if os.path.exists(DET_MODEL_PATH):
    yolo_model = YOLO(DET_MODEL_PATH)
    print(f"✅ YOLOv11 Detect Loaded: {DET_MODEL_PATH}")
else:
    print("❌ YOLO Detect Model not found!")

# 2. YOLOv8 Classification
cls_model = None
if os.path.exists(CLS_MODEL_PATH):
    cls_model = YOLO(CLS_MODEL_PATH)
    print(f"✅ YOLOv8 Classify Loaded: {CLS_MODEL_PATH}")
else:
    print("⚠️ Classify Model not found! Defaulting to 'asphalt'.")

# 3. SAM 2
sam2_checkpoint = "sam2_hiera_small.pt"
model_cfg = "sam2_hiera_s.yaml"

try:
    sam2_model = build_sam2(model_cfg, sam2_checkpoint, device=device)
    sam2_predictor = SAM2ImagePredictor(sam2_model)
    print("✅ SAM 2 Loaded")
except Exception as e:
    print(f"❌ SAM 2 Error: {e}")

## 4️⃣ Run Hybrid Inference

In [ ]:
import cv2
from tqdm import tqdm
import pandas as pd
import numpy as np
import shutil

# ===========================
# 0. Compute Homography
# ===========================
H_matrix, H_scale = calculate_homography(CALIB_IMAGE_PATH, MANUAL_CALIB_POINTS, ROAD_WIDTH_M)

# ===========================
# 1. Load GPS Data (Pre-load)
# ===========================
gps_df = None
video_start_time = None
if os.path.exists(GPS_LOG_PATH):
    try:
        gps_df = pd.read_csv(GPS_LOG_PATH)
        gps_df.columns = gps_df.columns.str.strip().str.lower()
        time_col = next((c for c in gps_df.columns if 'time' in c or 'date' in c), None)
        if time_col:
            gps_df[time_col] = pd.to_datetime(gps_df[time_col], format='mixed', utc=True)
            gps_df = gps_df.sort_values(time_col).rename(columns={time_col: 'time'})
            video_start_time = gps_df['time'].iloc[0]
            print(f"✅ GPS Loaded: {len(gps_df)} points")
    except Exception as e:
        print(f"⚠️ GPS Error: {e}")

# ===========================
# 2. Setup Video & Writer
# ===========================
cap = cv2.VideoCapture(VIDEO_PATH)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = cap.get(cv2.CAP_PROP_FPS)
width, height = int(cap.get(3)), int(cap.get(4))

os.makedirs("results/frames", exist_ok=True)
video_out = None
if GENERATE_VIDEO:
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out_fps = max(1, int(fps / FRAME_SKIP))
    video_out = cv2.VideoWriter("results/output_video.mp4", fourcc, out_fps, (width, height))

track_buffer = {}
final_detections = []
frame_idx = 0

pbar = tqdm(total=total_frames // FRAME_SKIP, desc="Pothole Processing")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break
    
    if frame_idx % FRAME_SKIP == 0:
        # Detect
        if USE_TRACKING:
            results = yolo_model.track(frame, persist=True, conf=CONFIDENCE_THRESHOLD, verbose=False)[0]
        else:
            results = yolo_model.predict(frame, conf=CONFIDENCE_THRESHOLD, verbose=False)[0]
            
        boxes = results.boxes.xyxy.cpu().numpy()
        track_ids = results.boxes.id.cpu().numpy().astype(int) if results.boxes.id is not None else [-1]*len(boxes)
        confs = results.boxes.conf.cpu().numpy()

        annotated_frame = frame.copy()

        if len(boxes) > 0:
            # Segment
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            sam2_predictor.set_image(frame_rgb)
            masks, scores, _ = sam2_predictor.predict(point_coords=None, point_labels=None, box=boxes, multimask_output=False)
            if masks.ndim == 4:
                if masks.shape[1] > 1:
                     masks = masks[np.arange(masks.shape[0]), np.argmax(scores, axis=1)]
                else:
                     masks = masks.squeeze(1)

            for i, (box, mask, conf, tid) in enumerate(zip(boxes, masks, confs, track_ids)):
                area_m2 = calculate_area_hybrid(mask, width, height, H=H_matrix, scale=H_scale)
                
                # GPS
                lat, lon = None, None
                if gps_df is not None and video_start_time is not None:
                    curr_time = video_start_time + pd.to_timedelta(frame_idx / fps, unit='s')
                    idx_near = (gps_df['time'] - curr_time).abs().idxmin()
                    row = gps_df.loc[idx_near]
                    lat, lon = float(row['latitude']), float(row['longitude'])

                # Classify
                pavement_type = "asphalt"
                if cls_model:
                    x1, y1, x2, y2 = map(int, box)
                    x1, y1 = max(0, x1), max(0, y1)
                    x2, y2 = min(width, x2), min(height, y2)
                    if x2 > x1 and y2 > y1:
                        crop = frame[y1:y2, x1:x2]
                        cls_res = cls_model.predict(crop, verbose=False)[0]
                        pavement_type = cls_res.names[cls_res.probs.top1]

                # Draw
                x1, y1, x2, y2 = box.astype(int)
                cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), (0,0,255), 2)
                # Red Mask
                m_u8 = (mask > 0).astype(np.uint8) * 255
                overlay = np.zeros_like(frame)
                overlay[m_u8 > 0] = (0,0,255)
                annotated_frame = cv2.addWeighted(annotated_frame, 1, overlay, 0.5, 0)
                
                label = f"ID:{tid}|{pavement_type}|{area_m2:.2f}m2"
                cv2.putText(annotated_frame, label, (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,255,255), 2)

                rec = {
                    'track_id': tid, 'area_m2': area_m2, 'pavement_type': pavement_type,
                    'confidence': conf, 'frame_idx': frame_idx, 'latitude': lat, 'longitude': lon,
                    'image_path': f"pothole_{tid}_{frame_idx}.jpg"
                }
                
                if USE_TRACKING and tid != -1:
                    if tid not in track_buffer: track_buffer[tid] = []
                    track_buffer[tid].append({'data': rec, 'img': annotated_frame, 'path': f"results/frames/{rec['image_path']}"})
                else:
                    cv2.imwrite(f"results/frames/{rec['image_path']}", annotated_frame)
                    final_detections.append(rec)
        
        if GENERATE_VIDEO and video_out: video_out.write(annotated_frame)
        pbar.update(1)
    frame_idx += 1

cap.release()
if video_out: video_out.release()
pbar.close()

# Post-Process
if USE_TRACKING:
    for tid, history in track_buffer.items():
        if not history: continue
        areas = [h['data']['area_m2'] for h in history]
        avg_area = np.mean(areas)
        best = min(history, key=lambda x: abs(x['data']['area_m2'] - avg_area))
        cv2.imwrite(best['path'], best['img'])
        final_detections.append(best['data'])

if final_detections:
    pd.DataFrame(final_detections).to_csv("results/detections.csv", index=False)
    shutil.make_archive("final_results", 'zip', "results")
    from google.colab import files
    files.download("final_results.zip")
    print("✅ Done.")